# Big Five Personality Prediction: LLaMA-3 & Mistral on Pandora

## Setup Instructions (READ FIRST)
1. Go to **Runtime → Change runtime type → T4 GPU** before running anything
2. Run cells **in order** — do not skip
3. `SAMPLE_SIZE` below controls how many users to evaluate. Start with 50 to test, then increase.

## Bugs Fixed from Original
- **LLaMA `temperature=0` crash**: `do_sample=False` conflicts with `temperature=0` in newer transformers. Fixed by removing `temperature` when `do_sample=False`.
- **LLaMA output parsing**: `decoded[len(prompt):]` is unreliable because the tokenizer may add special tokens that shift the offset. Fixed to decode only new tokens (same method Mistral used).
- **No sample limit**: The full test set can be thousands of entries — at ~2s/inference on a T4, that's hours. Added `SAMPLE_SIZE` cap.
- **No progress tracking**: Added `tqdm` so you can see it's actually running and not stuck.
- **No checkpointing**: If Colab disconnects mid-run, all results are lost. Added per-trait result saving.
- **Both models loaded simultaneously**: LLaMA (~16GB) + Mistral (~14GB) exceeds T4 VRAM (16GB). Fixed to delete/free LLaMA before loading Mistral.
- **Binarize threshold**: Original uses `threshold=0.0`, meaning scores of exactly 0 become 0 but any positive score becomes 1. This may produce severe class imbalance depending on the dataset. Noted with a warning.

In [ ]:
# ============================================================
# CELL 1: Install dependencies
# ============================================================
!pip install -q transformers accelerate datasets tqdm scikit-learn
# Note: 'accelerate' is required for device_map='auto' to work correctly

In [ ]:
# ============================================================
# CELL 2: Imports
# ============================================================
import torch
import gc
import json
import os
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from sklearn.metrics import classification_report, accuracy_score, f1_score
from tqdm import tqdm

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# ============================================================
# CELL 3: Configuration — ADJUST THESE
# ============================================================

# Start with 50 to verify everything runs, then increase to 200-500 for paper results
# Full test set may be 1000+ entries — at ~2s each that's 30+ min per model per trait
SAMPLE_SIZE = 100  # set to None to use full test set

USE_ENRICHED_PROMPT = True  # True = enriched (complex) prompt, False = simple prompt

RANDOM_SEED = 42  # for reproducible sampling

RESULTS_DIR = "/content/results"  # where to save intermediate results
os.makedirs(RESULTS_DIR, exist_ok=True)

TRAITS = ["Openness", "Conscientiousness", "Extraversion", "Agreeableness", "Neuroticism"]

In [ ]:
# ============================================================
# CELL 4: Load and prepare dataset
# ============================================================
print("Loading dataset...")
dataset_hf = load_dataset("Fatima0923/Automated-Personality-Prediction")
print(dataset_hf)
print("Columns:", dataset_hf["train"].column_names)

# Check score distributions before binarizing
# IMPORTANT: threshold=0.0 means any score > 0 becomes class 1.
# Depending on how scores are distributed, this can cause severe imbalance.
# Check the distribution before proceeding.
test_df = dataset_hf["test"].to_pandas()
print("\nScore distributions (test set):")
for trait in ["openness", "conscientiousness", "extraversion", "agreeableness", "neuroticism"]:
    if trait in test_df.columns:
        print(f"  {trait}: min={test_df[trait].min():.2f}, max={test_df[trait].max():.2f}, mean={test_df[trait].mean():.2f}")
        pct_positive = (test_df[trait] > 0.0).mean()
        print(f"    → {pct_positive*100:.1f}% will be class 1 with threshold=0.0")

In [ ]:
# ============================================================
# CELL 5: Binarize and convert dataset
# ============================================================
def binarize(x, threshold=0.0):
    return 1 if x > threshold else 0

def convert_dataset(hf_split):
    dataset = []
    for row in hf_split:
        dataset.append({
            "comments": [row["text"]],
            "labels": {
                "Openness": binarize(row["openness"]),
                "Conscientiousness": binarize(row["conscientiousness"]),
                "Extraversion": binarize(row["extraversion"]),
                "Agreeableness": binarize(row["agreeableness"]),
                "Neuroticism": binarize(row["neuroticism"]),
            }
        })
    return dataset

full_dataset = convert_dataset(dataset_hf["test"])

# Apply sample limit
if SAMPLE_SIZE is not None:
    import random
    random.seed(RANDOM_SEED)
    dataset = random.sample(full_dataset, min(SAMPLE_SIZE, len(full_dataset)))
    print(f"Using {len(dataset)} samples (out of {len(full_dataset)} total)")
else:
    dataset = full_dataset
    print(f"Using full dataset: {len(dataset)} samples")

# Show class balance in sample
print("\nClass balance in sample:")
for trait in TRAITS:
    n_pos = sum(1 for d in dataset if d["labels"][trait] == 1)
    print(f"  {trait}: {n_pos}/{len(dataset)} positive ({n_pos/len(dataset)*100:.1f}%)")

In [ ]:
# ============================================================
# CELL 6: Shared helpers
# ============================================================
def aggregate_user_comments(user_comments, max_chars=3000):
    # Reduced from 5000 to 3000 to avoid token limit issues on T4
    text = " ".join(user_comments)
    return text[:max_chars]

def parse_output(output):
    output = output.strip()
    if output.startswith("1"):
        return 1
    elif output.startswith("0"):
        return 0
    return None  # invalid output

trait_descriptions = {
    "Openness": "curiosity, imagination, preference for novelty",
    "Conscientiousness": "organization, discipline, reliability",
    "Extraversion": "sociability, assertiveness, energy",
    "Agreeableness": "empathy, cooperation, kindness",
    "Neuroticism": "emotional instability, anxiety, mood swings"
}

def save_results(results, model_name):
    """Save results to disk so they survive Colab disconnects."""
    path = os.path.join(RESULTS_DIR, f"{model_name}_results.json")
    serializable = {k: {"y_true": v[0], "y_pred": v[1]} for k, v in results.items()}
    with open(path, "w") as f:
        json.dump(serializable, f)
    print(f"Results saved to {path}")

def print_summary(results, model_label):
    print(f"\n{'='*50}")
    print(f"SUMMARY: {model_label}")
    print(f"{'='*50}")
    for trait in TRAITS:
        if trait not in results:
            continue
        y_true, y_pred = results[trait]
        print(f"\n--- {trait} ---")
        print(classification_report(y_true, y_pred, digits=3))
        print(f"Accuracy: {accuracy_score(y_true, y_pred):.3f}")
        print(f"F1 (macro): {f1_score(y_true, y_pred, average='macro'):.3f}")

---
# Part 1: LLaMA-3-8B-Instruct

In [ ]:
# ============================================================
# CELL 7: Load LLaMA
# ============================================================
# Note: You may need to accept the model license on HuggingFace first:
# https://huggingface.co/NousResearch/Meta-Llama-3-8B-Instruct
# Then run: huggingface-cli login

llama_model_name = "NousResearch/Meta-Llama-3-8B-Instruct"

print("Loading LLaMA tokenizer...")
llama_tokenizer = AutoTokenizer.from_pretrained(llama_model_name)

print("Loading LLaMA model (this may take 3-5 minutes)...")
llama_model = AutoModelForCausalLM.from_pretrained(
    llama_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
llama_model.eval()
print("LLaMA loaded.")
print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ============================================================
# CELL 8: LLaMA prompt builders
# ============================================================
def build_llama_simple_prompt(text, trait):
    return (
        f'You are a psychological text classifier.\n\n'
        f'Your task is to determine whether the Big Five trait "{trait}" is expressed in the TEXT.\n\n'
        f'Rules:\n'
        f'- Output ONLY "1" if the trait is present\n'
        f'- Output ONLY "0" if the trait is not present\n'
        f'- Do NOT explain\n'
        f'- Do NOT output anything else\n\n'
        f'TEXT:\n{text}\n\nAnswer:'
    )

def build_llama_enriched_prompt(text, trait):
    return (
        f'You are an expert in personality psychology using the Big Five model.\n\n'
        f'Determine whether the trait "{trait}" is present in the TEXT.\n\n'
        f'Trait definition: {trait_descriptions[trait]}\n\n'
        f'Guidelines:\n'
        f'- Look at tone, wording, emotional expression, and behavior\n'
        f'- Consider both explicit and implicit signals\n\n'
        f'Output ONLY one digit: "1" if present, "0" if absent.\n\n'
        f'TEXT:\n{text}\n\nAnswer:'
    )

In [ ]:
# ============================================================
# CELL 9: LLaMA inference
# BUG FIX 1: Removed temperature=0 (conflicts with do_sample=False in new transformers)
# BUG FIX 2: Decode only new tokens instead of slicing by prompt length
# ============================================================
@torch.no_grad()
def predict_llama(text, trait, enriched=True):
    prompt = build_llama_enriched_prompt(text, trait) if enriched else build_llama_simple_prompt(text, trait)

    inputs = llama_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(llama_model.device)

    outputs = llama_model.generate(
        **inputs,
        max_new_tokens=5,
        do_sample=False,
        # FIX: do NOT pass temperature=0 with do_sample=False — causes ValueError in transformers >= 4.38
        pad_token_id=llama_tokenizer.eos_token_id
    )

    # FIX: decode only the newly generated tokens, not the full sequence
    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    decoded = llama_tokenizer.decode(new_tokens, skip_special_tokens=True)

    return parse_output(decoded)

In [ ]:
# ============================================================
# CELL 10: Run LLaMA evaluation
# ============================================================
llama_results = {}
invalid_counts = {}

for trait in TRAITS:
    y_true = []
    y_pred = []
    n_invalid = 0

    print(f"\nEvaluating {trait}...")
    for user in tqdm(dataset, desc=trait):
        text = aggregate_user_comments(user["comments"])
        pred = predict_llama(text, trait, enriched=USE_ENRICHED_PROMPT)

        if pred is None:
            n_invalid += 1
            continue

        y_true.append(user["labels"][trait])
        y_pred.append(pred)

    llama_results[trait] = (y_true, y_pred)
    invalid_counts[trait] = n_invalid
    print(f"  Invalid outputs: {n_invalid}/{len(dataset)} ({n_invalid/len(dataset)*100:.1f}%)")
    print(classification_report(y_true, y_pred, digits=3))

    # Save after each trait in case of crash
    save_results(llama_results, "llama")

print_summary(llama_results, "LLaMA-3-8B-Instruct")

In [ ]:
# ============================================================
# CELL 11: Free LLaMA from VRAM before loading Mistral
# BUG FIX: Original loads both models simultaneously — exceeds T4 VRAM
# ============================================================
print(f"VRAM before cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB")

del llama_model
del llama_tokenizer
gc.collect()
torch.cuda.empty_cache()

print(f"VRAM after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print("LLaMA freed. Ready to load Mistral.")

---
# Part 2: Mistral-7B-Instruct

In [ ]:
# ============================================================
# CELL 12: Load Mistral
# ============================================================
mistral_model_name = "mistralai/Mistral-7B-Instruct-v0.3"

print("Loading Mistral tokenizer...")
mistral_tokenizer = AutoTokenizer.from_pretrained(mistral_model_name)

print("Loading Mistral model (3-5 minutes)...")
mistral_model = AutoModelForCausalLM.from_pretrained(
    mistral_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
mistral_model.eval()
print("Mistral loaded.")
print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ============================================================
# CELL 13: Mistral prompt builders (chat-template format)
# ============================================================
def build_mistral_simple_prompt(text, trait):
    return [{
        "role": "user",
        "content": (
            f'You are a psychological text classifier.\n\n'
            f'Determine whether the Big Five trait "{trait}" is expressed in the TEXT.\n\n'
            f'Output ONLY "1" if present, "0" if not. No explanation.\n\n'
            f'TEXT:\n{text}'
        )
    }]

def build_mistral_enriched_prompt(text, trait):
    return [{
        "role": "user",
        "content": (
            f'You are an expert in personality psychology using the Big Five model.\n\n'
            f'Determine whether the trait "{trait}" is present in the TEXT.\n\n'
            f'Trait definition: {trait_descriptions[trait]}\n\n'
            f'Guidelines: Look at tone, wording, emotional expression, and behavior. '
            f'Consider both explicit and implicit signals.\n\n'
            f'Output ONLY one digit: "1" if present, "0" if absent.\n\n'
            f'TEXT:\n{text}'
        )
    }]

In [ ]:
# ============================================================
# CELL 14: Mistral inference (already correct in original — kept as-is)
# ============================================================
@torch.no_grad()
def predict_mistral(text, trait, enriched=True):
    messages = build_mistral_enriched_prompt(text, trait) if enriched else build_mistral_simple_prompt(text, trait)

    encoded = mistral_tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True,
        truncation=True,
        max_length=1024
    ).to(mistral_model.device)

    outputs = mistral_model.generate(
        encoded,
        max_new_tokens=5,
        do_sample=False,
        temperature=None,
        top_p=None,
        pad_token_id=mistral_tokenizer.eos_token_id
    )

    new_tokens = outputs[0][encoded.shape[-1]:]
    decoded = mistral_tokenizer.decode(new_tokens, skip_special_tokens=True)
    return parse_output(decoded)

In [ ]:
# ============================================================
# CELL 15: Run Mistral evaluation
# ============================================================
mistral_results = {}
mistral_invalid_counts = {}

for trait in TRAITS:
    y_true = []
    y_pred = []
    n_invalid = 0

    print(f"\nEvaluating {trait}...")
    for user in tqdm(dataset, desc=trait):
        text = aggregate_user_comments(user["comments"])
        pred = predict_mistral(text, trait, enriched=USE_ENRICHED_PROMPT)

        if pred is None:
            n_invalid += 1
            continue

        y_true.append(user["labels"][trait])
        y_pred.append(pred)

    mistral_results[trait] = (y_true, y_pred)
    mistral_invalid_counts[trait] = n_invalid
    print(f"  Invalid outputs: {n_invalid}/{len(dataset)} ({n_invalid/len(dataset)*100:.1f}%)")
    print(classification_report(y_true, y_pred, digits=3))

    save_results(mistral_results, "mistral")

print_summary(mistral_results, "Mistral-7B-Instruct-v0.3")

In [ ]:
# ============================================================
# CELL 16: Side-by-side comparison table
# ============================================================
print("\n" + "="*60)
print("COMPARISON TABLE: LLaMA vs Mistral (Enriched Prompt, Pandora)")
print("="*60)
print(f"{'Trait':<20} {'LLaMA Acc':>10} {'LLaMA F1':>10} {'Mistral Acc':>12} {'Mistral F1':>10}")
print("-"*60)

for trait in TRAITS:
    la, lp = llama_results.get(trait, ([], []))
    ma, mp = mistral_results.get(trait, ([], []))

    l_acc = accuracy_score(la, lp) if la else float('nan')
    l_f1  = f1_score(la, lp, average='macro') if la else float('nan')
    m_acc = accuracy_score(ma, mp) if ma else float('nan')
    m_f1  = f1_score(ma, mp, average='macro') if ma else float('nan')

    print(f"{trait:<20} {l_acc:>10.3f} {l_f1:>10.3f} {m_acc:>12.3f} {m_f1:>10.3f}")

print("\nPre-LLM SOTA on Pandora (Di Cursi et al. 2025): ~0.50-0.55 macro-F1 (no config consistently exceeds)")
print("Results saved to:", RESULTS_DIR)